# 04 — Unsupervised Clustering: Validating the EU_REGIONS Grouping

`02_eda.ipynb` grouped countries into four regions (Western Europe/Nordics, Mediterranean, Eastern Europe, Baltics) purely by geographic and cultural proximity — a judgment call, not something derived from the data. This notebook asks the data directly: if you cluster the 27 EU countries on their socioeconomic and mental-health profiles with no knowledge of that grouping, does anything resembling it emerge?

This is also a standalone unsupervised analysis in its own right, independent of whether it confirms the regions: how many natural groups of countries does the data actually support, and what characterizes them?

**Result up front:** the four-region split gets moderate-to-good empirical support (ARI ≈ 0.57, NMI ≈ 0.72) — better than chance, not a perfect match. But the data's own preferred number of clusters, by silhouette score, is 2, not 4 — and that 2-way split doesn't track geography or suicide rate at all; it tracks economic scale. Both findings are reported, not just the one that confirms the original assumption.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

from src import (
    ID_COLS,
    TARGET,
    EU_REGIONS,
    build_predictor_list,
    aggregate_country_features,
    sweep_kmeans,
    run_kmeans,
    run_hierarchical,
    cluster_region_agreement,
    plot_kmeans_elbow_silhouette,
    plot_dendrogram,
    plot_cluster_vs_region_pca,
    plot_suicide_trend_by_group,
    save_figure,
)

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
np.random.seed(42)

fig_prefix = "04_"

df_development = pd.read_parquet("../data/processed/df_development.parquet")
predictor_features = build_predictor_list(df_development, ID_COLS, TARGET)
print(
    f"df_development: {df_development.shape[0]} rows | {len(predictor_features)} predictors"
)

## From country-year panel to one row per country

`Region` is a fixed property of a country, but `df_development` has one row per country *per year*. Clustering the 594 country-year rows directly would let the same country land in different clusters across years, which makes no sense for validating a label that never changes.

`aggregate_country_features()` collapses the panel to 27 rows — one per country — with:

- **Level**: the mean of every predictor and of `Suicide rate` over 2000-2021 — each country's typical socioeconomic/mental-health profile.
- **Trend**: the linear slope of `Suicide rate` over time — a country with a flat rate of 15 and one climbing from 5 toward 15 would look identical under level alone; the trend term tells them apart.

Only the target gets a trend term, not all 18 predictors — adding 18 more slope features for 27 countries would raise dimensionality well past what that many samples can support, and the trend that actually matters for this question is in the outcome variable, not in how fast GDP happened to move.

In [ ]:
country_df = aggregate_country_features(
    df_development, predictor_features, target=TARGET
)
print(f"Aggregated to {country_df.shape[0]} countries, {country_df.shape[1]} columns")
display(country_df.head())

feature_cols = [c for c in country_df.columns if c != "Code"]
scaler = RobustScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(country_df[feature_cols]),
    columns=feature_cols,
    index=country_df.index,
)
region_labels = country_df["Code"].map(EU_REGIONS)

## How many clusters does the data actually support?

Before comparing anything to the 4-region assumption, check what the data itself suggests — via the elbow method (where adding another cluster stops meaningfully reducing within-cluster variance) and the silhouette score (cluster separation quality, -1 to 1, higher is better). Testing k=4 without first checking this would risk finding whatever the region split predetermined, rather than what's actually there.

In [ ]:
sweep = sweep_kmeans(X_scaled, k_range=range(2, 9))
display(sweep)

fig = plot_kmeans_elbow_silhouette(sweep)
save_figure(fig, name="kmeans_elbow_silhouette", prefix=fig_prefix)

**The silhouette score peaks at k=2 (≈0.32), not at k=4** (≈0.20) — the four-region split is not what the data would choose on its own if asked for the "best" number of clusters. This doesn't rule out four groups being a reasonable and useful split; it means four is a finer-grained cut than the strongest structure in the data, which is coarser. Both k=2 and k=4 are examined below.

## What does the natural k=2 split actually represent?

Worth checking directly rather than assuming it must track suicide rate or geography, since neither was used to build it.

In [ ]:
labels_k2, _ = run_kmeans(X_scaled, k=2)
comparison_k2 = pd.DataFrame(
    {
        "Code": country_df["Code"],
        "Region": region_labels,
        "Cluster (k=2)": labels_k2,
        "Suicide rate (mean)": country_df["Suicide rate_mean"].round(1),
        "GDP per capita (mean)": country_df["GDP per capita_mean"].round(0),
        "Population (mean)": country_df["Population_mean"].round(0),
    }
)
display(comparison_k2.sort_values("Cluster (k=2)"))

The k=2 split groups Germany, France, Spain, and Italy against the other 23 countries. This tracks neither region nor suicide rate — Spain and Italy have some of the lowest rates in the dataset, Germany and France sit mid-to-high. What those four do share is population and economic scale: they are the EU's four largest economies by a wide margin. The strongest signal in this feature space at the coarsest level is country size, not suicide-relevant regional structure — worth keeping in mind before treating any cluster split as automatically meaningful for the suicide-rate question specifically.

## K-Means and hierarchical clustering at k=4

Now the actual comparison: cutting at k=4, matching the number of regions, with two different algorithms — if both independently reach a similar partition, that partition is more likely to reflect real structure than an artifact of one specific method.

In [ ]:
labels_kmeans, kmeans_model = run_kmeans(X_scaled, k=4)
linkage_matrix, labels_hier = run_hierarchical(X_scaled, k=4)

fig = plot_dendrogram(linkage_matrix, labels=country_df["Code"].tolist(), k=4)
save_figure(fig, name="dendrogram_k4", prefix=fig_prefix)

## Agreement with the a priori EU_REGIONS

Adjusted Rand Index (ARI) and Normalized Mutual Information (NMI) — both compare two partitions of the same items without requiring the cluster numbers to match the region names; a cluster labeled "2" can be a perfect match for "Baltics" without ever being told that name. Both range 0 (no agreement beyond chance) to 1 (perfect agreement). ARI penalizes any disagreement more strictly; NMI is more forgiving of a cluster that splits one region into two sub-groups without mixing different regions together.

In [ ]:
agreement_kmeans = cluster_region_agreement(labels_kmeans, region_labels)
agreement_hier = cluster_region_agreement(labels_hier, region_labels)

print(
    f"K-Means (k=4)     vs EU_REGIONS — ARI: {agreement_kmeans['ARI']:.3f} | NMI: {agreement_kmeans['NMI']:.3f}"
)
print(
    f"Hierarchical (k=4) vs EU_REGIONS — ARI: {agreement_hier['ARI']:.3f} | NMI: {agreement_hier['NMI']:.3f}"
)

**ARI ≈ 0.57, NMI ≈ 0.72 for both algorithms** — moderate-to-good agreement, well above what random cluster assignment would produce (≈0), but clearly short of a perfect match (1.0). Both algorithms landing on the same agreement level independently is itself informative: it isn't one method's idiosyncrasy, it's a property of the feature space. The geographic/cultural grouping used in `02_eda.ipynb` captures something real in the socioeconomic and mental-health data, but it isn't the single dominant structure — some countries' actual profile sits closer to a different region's typical member than to their own geographic neighbors.

## Temporal evolution: a priori regions vs data-driven clusters

The ARI/NMI numbers above summarize agreement as a single number; this section shows the same comparison as a trend over time — the a priori `EU_REGIONS` grouping and the K-Means cluster, each used to group the full country-year panel and plot suicide rate evolution 2000-2021. The cluster label is mapped back from `country_df` (one row per country) onto every year of that country in `df_development`, since a country's cluster is a fixed attribute, not something that varies by year.

In [ ]:
df_development["Region"] = df_development["Code"].map(EU_REGIONS)
cluster_map = dict(zip(country_df["Code"], labels_kmeans))
df_development["Cluster"] = df_development["Code"].map(cluster_map).apply(lambda c: f"Cluster {c}")

fig = plot_suicide_trend_by_group(
    df_development, "Region", legend_title="EU region (a priori)",
    title="Suicide rate evolution by a priori EU region (2000-2021)",
)
save_figure(fig, name="suicide_trend_by_region_a_priori", prefix=fig_prefix)

In [ ]:
fig = plot_suicide_trend_by_group(
    df_development, "Cluster", legend_title="K-Means cluster",
    title="Suicide rate evolution by data-driven cluster (2000-2021)",
)
save_figure(fig, name="suicide_trend_by_cluster", prefix=fig_prefix)

Reading the two side by side: the cluster trend lines should broadly echo the region trend lines wherever ARI/NMI showed agreement (Baltics, Eastern Europe), and diverge or mix wherever it didn'''t (the Mediterranean / Western Europe-Nordics boundary) — this is the same finding as the PCA scatter and per-cluster breakdown below, just viewed through time instead of through a 2D projection.

## Visual comparison: cluster assignment vs region, side by side

PCA reduces the 20-dimensional scaled feature space to 2 components purely for visualization — the axes have no direct real-world meaning beyond "the two directions of greatest variance in this feature set.

In [ ]:
fig = plot_cluster_vs_region_pca(
    X_scaled, labels_kmeans, region_labels, country_df["Code"].tolist()
)
save_figure(fig, name="cluster_vs_region_pca", prefix=fig_prefix)

## Which countries disagree, specifically

Rather than stopping at a summary statistic, listing exactly which countries the clustering assigns differently from their geographic region is what actually goes in the thesis discussion — the ARI/NMI numbers say *how much* disagreement, this says *where*.

In [ ]:
comparison_k4 = pd.DataFrame(
    {
        "Code": country_df["Code"],
        "Region (a priori)": region_labels,
        "Cluster (K-Means, k=4)": labels_kmeans,
    }
)

# For each cluster, show which regions its members actually belong to —
# a cluster mixing 3+ regions disagrees more than one mixing 2 adjacent ones.
for cluster_id in sorted(comparison_k4["Cluster (K-Means, k=4)"].unique()):
    members = comparison_k4[comparison_k4["Cluster (K-Means, k=4)"] == cluster_id]
    region_counts = members["Region (a priori)"].value_counts()
    print(f"Cluster {cluster_id}: {members['Code'].tolist()}")
    print(f"  Regions represented: {region_counts.to_dict()}\n")

## Where exactly the disagreement concentrates

Before the general conclusions, the exact per-cluster breakdown is worth showing, not just the aggregate ARI/NMI:

| Cluster | Countries | Region composition |
|---|---|---|
| 0 | EST, LTU, LVA | 100% Baltics |
| 1 | BGR, CZE, HRV, HUN, POL, ROU, SVK, SVN | 100% Eastern Europe |
| 2 | DEU, ESP, FRA, ITA | 2 Western Europe/Nordics + 2 Mediterranean |
| 3 | AUT, BEL, CYP, DNK, FIN, GRC, IRL, LUX, MLT, NLD, PRT, SWE | 8 Western Europe/Nordics + 4 Mediterranean |

**Baltics and Eastern Europe are recovered perfectly** — no country from either region appears outside its corresponding cluster. **All of the disagreement pulling ARI below 1.0 is concentrated exclusively at the Western Europe/Nordics – Mediterranean boundary** — neither region emerges as its own cluster; their countries are split and mixed across clusters 2 and 3.

One possible reading: socioeconomic variables (GDP per capita, health expenditure, life expectancy) separate Eastern Europe/Baltics from the rest far more cleanly than they separate Mediterranean from Western Europe/Nordics — the cultural distinction that motivated treating them as separate regions in `EU_REGIONS` may carry less weight in these specific variables than the dominant east-west economic divide.

---

## Conclusions

- The four-region grouping guessed initially is documented to have some relation to the data-driven 4-cluster partition at a moderate-to-good level (ARI ≈ 0.57, NMI ≈ 0.72), replicated by two different algorithms.
- The shown disagreement is not evenly distributed: Baltics and Eastern Europe are recovered perfectly; all the discrepancy concentrates at the Mediterranean / Western Europe-Nordics boundary, which the clustering does not treat as separate groups.
- It is also not the strongest structure available: silhouette score identifies k=2 as the best-separated partition, and that split tracks economic/population scale (Germany, France, Spain, Italy vs. the rest), not geography or suicide rate.
- Both can be true at once — geography/culture is a real axis of variation in this data without being the dominant one, and specifically the east-west divide explains more than the Mediterranean-Western split. For the thesis write-up: the regional grouping is defensible as a descriptive tool (it was never fed into the models as a predictor), but should be presented as a partially corroborated assumption, not a "discovered" structure — and if refined, merging Mediterranean and Western Europe/Nordics into a single region would have more empirical support than keeping them separate.
- Limitation worth flagging: with only 27 countries and ~20 aggregated features, any clustering result here has limited statistical power — a handful of reassigned countries would shift ARI/NMI substantially. Treat the decimal values as indicative, not precise.

## A note on using this cluster as a model feature

An earlier version of this pipeline fed the cluster identified here back into `03_models.ipynb` as a supervised feature (built in a leakage-safe way — no target information, fit on training data only). It was reverted after discussion: even done safely, adding it changes what the supervised models are actually explaining, and blurs together two separate results that are each cleaner read on their own — whether the socioeconomic/mental-health determinants predict suicide rate (`03_models.ipynb`), and whether the a priori regional grouping holds up in the data (this notebook). The leakage-safe functions (`fit_country_clusters`, `assign_country_clusters`, `add_cluster_feature` in `src/clustering.py`) are still there, tested, and available if a future version of this project wants to revisit that decision.